# Catálogo de dados CRUS — uma célula por API

Cada célula puxa **uma API** e mostra o dado **cru**, sem transformação:
sem renomear colunas, sem tratar valores ausentes (`-999`, `-`, `...`), sem agregar.
A única coisa feita é estruturar o JSON em tabela (`DataFrame`) pra dar pra ver.

Amostra: 10 municípios de PE.

In [1]:
# Setup: imports + helper + amostra (nome -> código IBGE)
import requests
import pandas as pd

def get_json(url, timeout=90):
    r = requests.get(url, timeout=timeout, headers={"Accept": "application/json"})
    r.raise_for_status()
    return r.json()

MUNICIPIOS = {
    "Petrolina": "2611101", "Araripina": "2601102", "Salgueiro": "2612208",
    "Serra Talhada": "2613909", "Arcoverde": "2601201", "Ouricuri": "2609907",
    "Custódia": "2605103", "Ibimirim": "2606606",
    "Santa Cruz do Capibaribe": "2612505", "Buíque": "2602803",
}
COD = list(MUNICIPIOS.values())
LOC = "N6[" + ",".join(COD) + "]"
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

In [2]:
# API 1 — Nominatim (OpenStreetMap): geocodificação nome -> lat/lon
# CRU: lista de resultados; json_normalize mostra todos os campos como vêm.
r = requests.get(
    "https://nominatim.openstreetmap.org/search",
    params={"q": "Petrolina, Pernambuco, Brasil", "format": "json", "limit": 3},
    headers={"User-Agent": "quantum-ibf/1.0 (castrobeneyto@gmail.com)"},
    timeout=30,
)
pd.json_normalize(r.json())

,place_id,licence,osm_type,osm_id,lat,lon,class,type,place_rank,importance,addresstype,name,display_name,boundingbox
0,15248847,"Data © OpenStreetMap contributors, ODbL 1.0. h...",relation,303629,-9.3817334,-40.4968875,boundary,administrative,16,0.509427,municipality,Petrolina,"Petrolina, Pernambuco, Região Nordeste, Brasil","[-9.4830959, -8.6234993, -40.9590393, -40.2080..."
1,15148414,"Data © OpenStreetMap contributors, ODbL 1.0. h...",relation,6207447,-9.2062082,-40.4255373,boundary,administrative,18,0.160058,city_district,Petrolina,"Petrolina, Pernambuco, Região Nordeste, Brasil","[-9.4830959, -8.9231977, -40.7751250, -40.2080..."


In [3]:
# API 2 — NASA POWER: clima diário por ponto (lat/lon de Petrolina)
# CRU: dict {parametro: {AAAAMMDD: valor}}. Janela curta (2023) só pra visualizar.
# NOTA: -999 = sem dado (NÃO tratei); índice = data AAAAMMDD como veio.
resp = get_json(
    "https://power.larc.nasa.gov/api/temporal/daily/point"
    "?parameters=PRECTOTCORR,T2M,T2M_MAX,T2M_MIN,RH2M&community=AG"
    "&latitude=-9.3817&longitude=-40.4969&start=20230101&end=20231231&format=JSON"
)
raw = pd.DataFrame(resp["properties"]["parameter"])
print("parâmetros (colunas):", list(raw.columns))
raw.head(10)

parâmetros (colunas): ['PRECTOTCORR', 'T2M', 'T2M_MAX', 'T2M_MIN', 'RH2M']


,PRECTOTCORR,T2M,T2M_MAX,T2M_MIN,RH2M
20230101,1.23,27.01,34.05,20.60,62.95
20230102,4.18,26.40,32.73,21.31,68.55
20230103,5.55,25.20,30.29,21.22,71.83
20230104,1.43,25.90,31.01,21.09,71.72
20230105,0.16,26.88,32.90,21.06,66.31
20230106,0.10,27.12,33.05,21.92,64.91
20230107,0.08,27.58,34.00,21.61,60.63
20230108,0.09,27.33,34.56,20.54,58.35
20230109,0.11,27.52,34.51,20.68,58.56
20230110,0.15,28.35,34.86,22.24,57.17


In [4]:
# API 3 — IBGE Localidades: municípios de PE (UF 26)
# CRU: nome, código (id) e hierarquia de região aninhada.
munis = get_json("https://servicodados.ibge.gov.br/api/v1/localidades/estados/26/municipios")
print("total de municípios em PE:", len(munis))
pd.json_normalize(munis).head()

total de municípios em PE: 185


,id,nome,microrregiao.id,microrregiao.nome,microrregiao.mesorregiao.id,microrregiao.mesorregiao.nome,microrregiao.mesorregiao.UF.id,microrregiao.mesorregiao.UF.sigla,microrregiao.mesorregiao.UF.nome,microrregiao.mesorregiao.UF.regiao.id,microrregiao.mesorregiao.UF.regiao.sigla,microrregiao.mesorregiao.UF.regiao.nome,regiao-imediata.id,regiao-imediata.nome,regiao-imediata.regiao-intermediaria.id,regiao-imediata.regiao-intermediaria.nome,regiao-imediata.regiao-intermediaria.UF.id,regiao-imediata.regiao-intermediaria.UF.sigla,regiao-imediata.regiao-intermediaria.UF.nome,regiao-imediata.regiao-intermediaria.UF.regiao.id,regiao-imediata.regiao-intermediaria.UF.regiao.sigla,regiao-imediata.regiao-intermediaria.UF.regiao.nome
0,2600054,Abreu e Lima,26017,Recife,2605,Metropolitana de Recife,26,PE,Pernambuco,2,NE,Nordeste,260001,Recife,2601,Recife,26,PE,Pernambuco,2,NE,Nordeste
1,2600104,Afogados da Ingazeira,26003,Pajeú,2601,Sertão Pernambucano,26,PE,Pernambuco,2,NE,Nordeste,260014,Afogados da Ingazeira,2603,Serra Talhada,26,PE,Pernambuco,2,NE,Nordeste
2,2600203,Afrânio,26005,Petrolina,2602,São Francisco Pernambucano,26,PE,Pernambuco,2,NE,Nordeste,260015,Petrolina,2604,Petrolina,26,PE,Pernambuco,2,NE,Nordeste
3,2600302,Agrestina,26012,Brejo Pernambucano,2603,Agreste Pernambucano,26,PE,Pernambuco,2,NE,Nordeste,260009,Caruaru,2602,Caruaru,26,PE,Pernambuco,2,NE,Nordeste
4,2600401,Água Preta,26015,Mata Meridional Pernambucana,2604,Mata Pernambucana,26,PE,Pernambuco,2,NE,Nordeste,260003,Palmares,2601,Recife,26,PE,Pernambuco,2,NE,Nordeste


In [5]:
# API 4 — IBGE Agregados 6579: população residente estimada (2024)
# CRU: JSON aninhado (variavel > resultados > series > serie). Achato só a estrutura;
# o valor fica exatamente como veio (string).
pop = get_json(
    "https://servicodados.ibge.gov.br/api/v3/agregados/6579/"
    f"periodos/2024/variaveis/9324?localidades={LOC}"
)
pd.DataFrame([
    {"cod_ibge": s["localidade"]["id"], "municipio": s["localidade"]["nome"],
     pop[0]["variavel"]: list(s["serie"].values())[0], "unidade": pop[0]["unidade"]}
    for s in pop[0]["resultados"][0]["series"]
])

,cod_ibge,municipio,População residente estimada,unidade
0,2611101,Petrolina (PE),414083,Pessoas
1,2601102,Araripina (PE),90104,Pessoas
2,2612208,Salgueiro (PE),65635,Pessoas
3,2613909,Serra Talhada (PE),98143,Pessoas
4,2601201,Arcoverde (PE),82003,Pessoas
5,2609907,Ouricuri (PE),68329,Pessoas
6,2605103,Custódia (PE),39403,Pessoas
7,2606606,Ibimirim (PE),28760,Pessoas
8,2612505,Santa Cruz do Capibaribe (PE),104277,Pessoas
9,2602803,Buíque (PE),54425,Pessoas


In [6]:
# API 5 — IBGE Agregados 5938: PIB municipal a preços correntes (2021)
# CRU: valor como veio (string), unidade "Mil Reais".
pib = get_json(
    "https://servicodados.ibge.gov.br/api/v3/agregados/5938/"
    f"periodos/2021/variaveis/37?localidades={LOC}"
)
pd.DataFrame([
    {"cod_ibge": s["localidade"]["id"], "municipio": s["localidade"]["nome"],
     pib[0]["variavel"]: list(s["serie"].values())[0], "unidade": pib[0]["unidade"]}
    for s in pib[0]["resultados"][0]["series"]
])

,cod_ibge,municipio,Produto Interno Bruto a preços correntes,unidade
0,2611101,Petrolina (PE),7994040,Mil Reais
1,2601102,Araripina (PE),1272092,Mil Reais
2,2612208,Salgueiro (PE),1002105,Mil Reais
3,2613909,Serra Talhada (PE),1693347,Mil Reais
4,2601201,Arcoverde (PE),1215383,Mil Reais
5,2609907,Ouricuri (PE),780513,Mil Reais
6,2605103,Custódia (PE),593880,Mil Reais
7,2606606,Ibimirim (PE),330119,Mil Reais
8,2612505,Santa Cruz do Capibaribe (PE),1925734,Mil Reais
9,2602803,Buíque (PE),549015,Mil Reais


In [7]:
# API 6 — IBGE Agregados 4714: Censo 2022 — área (km²) e densidade (hab/km²)
# CRU: duas variáveis na mesma chamada; monto a tabela sem converter os valores.
ad = get_json(
    "https://servicodados.ibge.gov.br/api/v3/agregados/4714/"
    f"periodos/2022/variaveis/6318|614?localidades={LOC}"
)
linhas = {}
for var in ad:
    for s in var["resultados"][0]["series"]:
        cod = s["localidade"]["id"]
        linhas.setdefault(cod, {"cod_ibge": cod, "municipio": s["localidade"]["nome"]})
        linhas[cod][f"{var['variavel']} ({var['unidade']})"] = list(s["serie"].values())[0]
pd.DataFrame(linhas.values())

,cod_ibge,municipio,Área da unidade territorial (Quilômetros quadrados),Densidade demográfica (Habitante por quilômetro quadrado)
0,2611101,Petrolina (PE),4561.870,84.79
1,2601102,Araripina (PE),2037.394,41.76
2,2612208,Salgueiro (PE),1678.564,37.16
3,2613909,Serra Talhada (PE),2980.007,30.95
4,2601201,Arcoverde (PE),343.923,226.04
5,2609907,Ouricuri (PE),2381.570,27.40
6,2605103,Custódia (PE),1404.126,26.85
7,2606606,Ibimirim (PE),1882.498,14.13
8,2612505,Santa Cruz do Capibaribe (PE),335.309,293.03
9,2602803,Buíque (PE),1336.544,38.98


In [8]:
# API 7 — IBGE PAM 1612: lavoura TEMPORÁRIA (milho, feijão, mandioca, cana...)
# CRU, formato LONGO: município × produto × variável. 33 produtos.
# vars: 109 = área plantada (ha), 215 = valor da produção (mil R$).
tmp = get_json(
    "https://servicodados.ibge.gov.br/api/v3/agregados/1612/"
    f"periodos/2023/variaveis/109|215?localidades={LOC}&classificacao=81[all]"
)
rows = []
for var in tmp:
    for res in var["resultados"]:
        (_, produto), = res["classificacoes"][0]["categoria"].items()
        for s in res["series"]:
            rows.append({"cod_ibge": s["localidade"]["id"], "produto": produto,
                         "variavel": var["variavel"], "valor": list(s["serie"].values())[0]})
temporarias = pd.DataFrame(rows)
print("linhas:", len(temporarias), "| produtos:", temporarias["produto"].nunique(),
      "| '-' e '...' = sem produção / não informado (NÃO tratei)")
temporarias.head(20)

linhas: 680 | produtos: 34 | '-' e '...' = sem produção / não informado (NÃO tratei)


,cod_ibge,produto,variavel,valor
0,2611101,Total,Área plantada,10810
1,2601102,Total,Área plantada,27623
2,2612208,Total,Área plantada,1210
3,2613909,Total,Área plantada,5433
4,2601201,Total,Área plantada,515
5,2609907,Total,Área plantada,16840
6,2605103,Total,Área plantada,7975
7,2606606,Total,Área plantada,2664
8,2612505,Total,Área plantada,330
9,2602803,Total,Área plantada,1335


In [9]:
# API 8 — IBGE PAM 1613: lavoura PERMANENTE (uva, manga, banana, café...)
# CRU, formato LONGO: município × produto × variável. 38 produtos.
# vars: 2313 = área destinada à colheita (ha), 215 = valor (mil R$).
# (permanentes NÃO têm "área plantada"/109 — usam 2313.)
perm = get_json(
    "https://servicodados.ibge.gov.br/api/v3/agregados/1613/"
    f"periodos/2023/variaveis/2313|215?localidades={LOC}&classificacao=82[all]"
)
rows = []
for var in perm:
    for res in var["resultados"]:
        (_, produto), = res["classificacoes"][0]["categoria"].items()
        for s in res["series"]:
            rows.append({"cod_ibge": s["localidade"]["id"], "produto": produto,
                         "variavel": var["variavel"], "valor": list(s["serie"].values())[0]})
permanentes = pd.DataFrame(rows)
print("linhas:", len(permanentes), "| produtos:", permanentes["produto"].nunique())
permanentes.head(20)

linhas: 780 | produtos: 39


,cod_ibge,produto,variavel,valor
0,2611101,Total,Área destinada à colheita,24345
1,2601102,Total,Área destinada à colheita,6
2,2612208,Total,Área destinada à colheita,148
3,2613909,Total,Área destinada à colheita,70
4,2601201,Total,Área destinada à colheita,...
5,2609907,Total,Área destinada à colheita,20
6,2605103,Total,Área destinada à colheita,...
7,2606606,Total,Área destinada à colheita,58
8,2612505,Total,Área destinada à colheita,1
9,2602803,Total,Área destinada à colheita,1385
